In [9]:
import os
import mne
import numpy as np
import pandas as pd
from scipy import interpolate
import matplotlib.pyplot as plt
from scipy.signal import resample
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [5]:
# root dir
root = 'MCEF-RS/'
# participants file path
participants_path = os.path.join(root, 'participants.tsv')
participants = pd.read_csv(participants_path, sep='\t')
participants

,participant_id,Gender,Age,NativeLanguage,EducationLevel,Handedness,Inclusion,Comment
0,sub-001,man,23,french,16,right-handed,1,NaN
1,sub-002,woman,25,french,17,right-handed,1,NaN
2,sub-003,man,22,french,15,right-handed,1,NaN
3,sub-004,man,23,french,18,right-handed,1,NaN
4,sub-005,woman,31,french,20,right-handed,1,NaN
...,...,...,...,...,...,...,...,...
160,sub-161,woman,34,french,16,right-handed,0,Panic attack during rest
161,sub-162,woman,20,french,15,right-handed,1,NaN
162,sub-163,man,32,french,15,right-handed,1,NaN
163,sub-164,man,29,french,17,right-handed,1,NaN


In [6]:
# build subject info: "sub-XXX" -> (label, pid)
sub_info = {}  # key: subject name, value: (label, pid)
for row in participants.itertuples(index=False):
    sub_name = row[0]          # e.g. 'sub-001'
    pid = int(sub_name[-3:])   # convert '001' -> 1
    label = 0               # default label 0
    sub_info[sub_name] = (label, pid)

len(sub_info), list(sub_info.items())[:5]

(165,
 [('sub-001', (0, 1)),
  ('sub-002', (0, 2)),
  ('sub-003', (0, 3)),
  ('sub-004', (0, 4)),
  ('sub-005', (0, 5))])

## Features

In [9]:
"""# Test for bad channels, sampling freq and shape
bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                # print(raw.get_montage())
                # get bad channels
                bad_channel = raw.info['bads']
                bad_channel_list.append(bad_channel)
                # get sampling frequency
                sampling_freq = raw.info['sfreq']
                sampling_freq_list.append(sampling_freq)
                # get eeg data
                data = raw.get_data()
                data_shape = data.shape
                data_shape_list.append(data_shape)"""

Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\MCEF-RS\MCEF-RS\sub-001\eeg\sub-001_task-restingstate_eeg.fdt
Reading 0 ... 156802  =      0.000 ...   306.254 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\MCEF-RS\MCEF-RS\sub-002\eeg\sub-002_task-restingstate_eeg.fdt
Reading 0 ... 156791  =      0.000 ...   306.232 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\MCEF-RS\MCEF-RS\sub-003\eeg\sub-003_task-restingstate_eeg.fdt
Reading 0 ... 156794  =      0.000 ...   306.238 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\MCEF-RS\MCEF-RS\sub-004\eeg\sub-004_task-restingstate_eeg.fdt
Reading 0 ... 156775  =      0.000 ...   306.201 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\MCEF-RS\MCEF-RS\sub-005\eeg\sub-005_task-restingstate_eeg.fdt
Reading 0 ... 156794  =      0.000 ...   306.238 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\MCEF-RS\MCEF-RS\sub-006\eeg\sub-006_task-restingstate_eeg

In [10]:
"""from collections import Counter

print(bad_channel_list)
print(data_shape_list[0])
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))"""

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
(64, 156803)
Channel number counter: Counter({64: 165})
Sampling rate counter: Counter({512.0: 164, 2048.0: 1})


## Data preprocessing

In [7]:
def data_preprocessing(
    raw: mne.io.Raw,
    montage: str,
    notch_freq: float = 50.0,
    l_freq: float = 0.5,
    h_freq: float = 45.0,
    verbose=True
):
    """
    Clean EEG data using bandpass filtering, percentile-based bad channel detection,
    ICA + ICLabel artifact removal, resampling, re-referencing, epoching, and z-score normalization.

    Args:
        raw (mne.io.Raw): MNE Raw object containing EEG data.
        montage (str): Montage name for the EEG data (e.g., 'biosemi128').
        notch_freq (float): Notch filter frequency (default: 50 Hz).
        l_freq (float): Low cutoff frequency for bandpass filter (default: 0.5 Hz).
        h_freq (float): High cutoff frequency for bandpass filter (default: 40 Hz).
        verbose (bool): Verbose output.

    Returns:
        np.ndarray: Cleaned, normalized EEG data, shape (n_epochs, time_steps, channels).
    """

    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage(montage))
    if verbose:
        print(f"✔ Montage set: {montage}.")
        
    # 3. Notch Filter at 50 Hz
    raw.notch_filter(freqs=notch_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Notch @ {notch_freq} Hz")

    # 4. Bandpass Filter (0.5–45 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
    if verbose:
        print(f"✔ Bandpass filter applied ({l_freq}–{h_freq} Hz).")
    
    # 5. Set average reference for ICA
    raw.set_eeg_reference('average', projection=False)
    if verbose:
        print("✔ EEG re-referenced (average) before ICA.")
        
    # ICA . Performance drop when using ICA, might be due to already preprocessed data in their paper.
    raw_ica = raw.copy()
    ica = ICA(n_components=0.99, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")
        
    return raw

In [8]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [10]:
# Loop through each subject and session to preprocess the EEG data
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
# For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
# So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")
sub_id = 1
for sub in os.listdir(root):
    if 'sub-' in sub:
        li_sub = []
        sub_path = os.path.join(root, sub, 'eeg/')
        print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                NEAREST_19_AB = ['A1','B1','A7','A5','B30','B5','B7','A15','A13','B13','B15','B17','A23','A21','A29','B23','B25','A25','B27']
                raw.pick(NEAREST_19_AB)  # pick channels that are nearest to 19 channels
                STANDARD_19 = ['Fp1','Fp2','F7','F3','Fz','F4','F8','T7','C3','Cz','C4','T8','P7','P3','Pz','P4','P8','O1','O2']
                raw.rename_channels({ch: STANDARD_19[i] for i, ch in enumerate(raw.ch_names)})
                original_fs = raw.info['sfreq']
                T_raw = raw.n_times
                for fs in SAMPLE_RATE_LIST:
                    T_res = int(T_raw * fs / original_fs)
                    # compute number of segments
                    n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                    subject_segment_counts[sub_id][fs] += n_seg
                    N_total += n_seg
                    print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                    print("----------------------------------------")
        sub_id += 1
    print("-------------------------------------\n")
    

print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

MCEF-RS/sub-001\eeg/
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\MCEF-RS\MCEF-RS\sub-001\eeg\sub-001_task-restingstate_eeg.fdt
Reading 0 ... 156802  =      0.000 ...   306.254 secs...
fs=200Hz: T_res=61251, STEP=200, n_seg=305
----------------------------------------
fs=100Hz: T_res=30625, STEP=200, n_seg=152
----------------------------------------
fs=50Hz: T_res=15312, STEP=200, n_seg=75
----------------------------------------
-------------------------------------

MCEF-RS/sub-002\eeg/
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\MCEF-RS\MCEF-RS\sub-002\eeg\sub-002_task-restingstate_eeg.fdt
Reading 0 ... 156791  =      0.000 ...   306.232 secs...
fs=200Hz: T_res=61246, STEP=200, n_seg=3

In [17]:
output_root = os.path.join('Processed', sub_folder_path, 'MCEF-RS')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\MCEF-RS\X.dat
y path: Processed\L400\MCEF-RS\y.dat


## PASS 2: Process and store segments into memmap

In [18]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
sub_id = 1
for sub in os.listdir(root):
    if 'sub-' in sub:
        li_sub = []
        sub_path = os.path.join(root, sub, 'eeg/')
        label, pid = sub_info[sub]
        print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                NEAREST_19_AB = ['A1','B1','A7','A5','B30','B5','B7','A15','A13','B13','B15','B17','A23','A21','A29','B23','B25','A25','B27']
                raw.pick(NEAREST_19_AB)  # pick channels that are nearest to 19 channels
                STANDARD_19 = ['Fp1','Fp2','F7','F3','Fz','F4','F8','T7','C3','Cz','C4','T8','P7','P3','Pz','P4','P8','O1','O2']
                raw.rename_channels({ch: STANDARD_19[i] for i, ch in enumerate(raw.ch_names)})
                raw = data_preprocessing(
                    raw=raw,
                    montage='standard_1020',  # original montage
                    notch_freq=50,   # see json file
                    l_freq=0.5,
                    h_freq=45.0,
                    verbose=True
                )
                original_fs = raw.info['sfreq']
                T_raw = raw.n_times
                total_seconds_all += T_raw / original_fs
                data_raw = raw.get_data().T  # (T_raw, C)
                for fs in SAMPLE_RATE_LIST:
                    # resample to target fs
                    data = resample_time_series(data_raw, original_fs, fs)
                    T_res, _ = data.shape
                    print(f"fs={fs}Hz: resampled shape={data.shape}")
        
                    # overlapped sliding window with fixed STEP (in timestamps)
                    starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                    print(f"fs={fs}Hz: segments={len(starts)}")
        
                    for s in starts:
                        if cur >= N_total:
                            raise RuntimeError("Exceeded predicted N_total.")
        
                        X_mm[cur] = data[s:s + SEG_LEN]   # (SEG_LEN, C)
                        y_mm[cur, 0] = float(label)       # label
                        y_mm[cur, 1] = float(sub_id)      # Global session ID
                        y_mm[cur, 2] = float(fs)          # sample rate (scale)
                        cur += 1
        sub_id += 1
    print("-------------------------------------\n")
    

total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

MCEF-RS/sub-001\eeg/
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\MCEF-RS\MCEF-RS\sub-001\eeg\sub-001_task-restingstate_eeg.fdt
Reading 0 ... 156802  =      0.000 ...   306.254 secs...
✔ Montage set: standard_1020.
✔ Notch @ 50 Hz
✔ Bandpass filter applied (0.5–45.0 Hz).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
✔ EEG re-referenced (average) before ICA.
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by explained variance: 15 components
Fitting ICA took 1.2s.
✔ ICA fitted.
Applying ICA to Raw instance
    Transforming to ICA space (15 components)
    Zeroing out 1 ICA component
    Projecting back using 19 PCA components
✔ ICA applie

## Load and check the processed data

In [19]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 88411
  T = 400
  C = 19
  X_path = Processed\L400\MCEF-RS\X.dat
  y_path = Processed\L400\MCEF-RS\y.dat
-------------------------------------
Subject ID 001: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 002: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 003: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 004: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 005: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 006: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 007: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 008: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 009: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 010: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 011: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 012: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 013: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 014: X shape=(532, 400, 19), y shape=(532, 3)
Subject ID 015: X shape=(532, 400, 19), y shape